# LSTM model using stock market prices

In this notebook, I will implement an LSTM model and train it based on stock market prices using data I get from Nasdaq.

- META: https://www.nasdaq.com/market-activity/stocks/meta/historical?page=1&rows_per_page=10&timeline=y10
- AMZN: https://www.nasdaq.com/market-activity/stocks/amzn/historical?page=1&rows_per_page=10&timeline=y10
- AAPL: https://www.nasdaq.com/market-activity/stocks/aapl/historical?page=1&rows_per_page=10&timeline=y10
- NFLX: https://www.nasdaq.com/market-activity/stocks/nflx/historical?page=1&rows_per_page=10&timeline=y10
- GOOGL: https://www.nasdaq.com/market-activity/stocks/googl/historical?page=1&rows_per_page=10&timeline=y10
- MSFT: https://www.nasdaq.com/market-activity/stocks/msft/historical?page=1&rows_per_page=10&timeline=y10

Although I have multiple data collected here, I will just focus on one data set (META) right now. Maybe I can do something later with more data sets.

Also, for this notebook, I will try to use the plotly package.

### Importing Packages and Libraries

In [205]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import DataLoader, TensorDataset


### Data Collection

In [218]:
df_META = pd.read_csv("data/META.csv")
df_AMZN = pd.read_csv("data/AMZN.csv")
df_AAPL = pd.read_csv("data/AAPL.csv")
df_NFLX = pd.read_csv("data/NFLX.csv")
df_GOOGL = pd.read_csv("data/GOOGL.csv")
df_MSFT = pd.read_csv("data/MSFT.csv")

print(df_META.shape)
df_META.head()

(2514, 6)


,Date,Close/Last,Volume,Open,High,Low
0,07/03/2025,$719.01,8601653,$726.61,$729.03,$714.42
1,07/02/2025,$713.57,9336740,$715.325,$720.30,$712.80
2,07/01/2025,$719.22,13431250,$736.875,$737.7499,$715.37
3,06/30/2025,$738.09,15402110,$744.55,$747.90,$734.25
4,06/27/2025,$733.63,18775740,$726.515,$735.43,$725.86


### Data Wrangling/Cleaning

In [219]:

df_META = df_META.rename(columns = {'Close/Last' : 'Close'})
df_AMZN = df_AMZN.rename(columns = {'Close/Last' : 'Close'})
df_AAPL = df_AAPL.rename(columns = {'Close/Last' : 'Close'})
df_NFLX = df_NFLX.rename(columns = {'Close/Last' : 'Close'})
df_GOOGL = df_GOOGL.rename(columns = {'Close/Last' : 'Close'})
df_MSFT = df_MSFT.rename(columns = {'Close/Last' : 'Close'})

collected_data = {"META":df_META, "AMZN":df_AMZN, "AAPL":df_AAPL, "NFLX": df_NFLX, "GOOGL": df_GOOGL, "MSFT": df_MSFT}

for temp_df in collected_data.values():
    temp_df['Close'] = temp_df['Close'].str[1:].astype(float)
    temp_df['Open'] = temp_df['Open'].str[1:].astype(float)
    temp_df['High'] = temp_df['High'].str[1:].astype(float)
    temp_df['Low'] = temp_df['Low'].str[1:].astype(float)
    temp_df['Date'] = pd.to_datetime(temp_df['Date'], format="%m/%d/%Y")

# df_META = df_META[df_META['Date'] > '2023-01-01']
# df_AAPL = df_AAPL[df_AAPL['Date'] > '2023-01-01']
df_GOOGL = df_GOOGL[df_GOOGL['Date'] > '2023-01-01']

df_GOOGL.tail()

,Date,Close,Volume,Open,High,Low
622,2023-01-09,88.02,29003900,88.360,90.05,87.860
623,2023-01-06,87.34,41381500,86.790,87.69,84.860
624,2023-01-05,86.20,27194380,87.470,87.57,85.900
625,2023-01-04,88.08,34854780,90.350,90.65,87.271
626,2023-01-03,89.12,28131220,89.585,91.05,88.520


### Data Visualization

In [220]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_META['Date'], y=df_META['Close'], name='Close', marker={'color':'yellow'}, mode='lines'))
fig.add_trace(go.Scatter(x=df_META['Date'], y=df_META['High'], name='High', marker={'color':'green'}, mode='lines'))
fig.add_trace(go.Scatter(x=df_META['Date'], y=df_META['Low'], name='Low', marker={'color':'red'}, mode='lines'))

fig.update_layout(title="META stock prices",
                  xaxis_title="Date",
                  yaxis_title="Stock price")

fig.show()


In [221]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_AMZN['Date'], y=df_AMZN['Low'], name='Low', marker={'color':'red'}, mode='lines'))
fig.update_layout(title="AMZN stock prices", xaxis_title="Date", yaxis_title="Stock price")

fig.show()

In [222]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_AAPL['Date'], y=df_AAPL['Low'], name='Low', marker={'color':'red'}, mode='lines'))
fig.update_layout(title="AAPL stock prices", xaxis_title="Date", yaxis_title="Stock price")

fig.show()

In [223]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_NFLX['Date'], y=df_NFLX['Low'], name='Low', marker={'color':'red'}, mode='lines'))
fig.update_layout(title="NFLX stock prices", xaxis_title="Date", yaxis_title="Stock price")

fig.show()

In [224]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_GOOGL['Date'], y=df_GOOGL['Low'], name='Low', marker={'color':'red'}, mode='lines'))
fig.update_layout(title="GOOGL stock prices", xaxis_title="Date", yaxis_title="Stock price")

fig.show()

In [225]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_MSFT['Date'], y=df_MSFT['Low'], name='Low', marker={'color':'red'}, mode='lines'))
fig.update_layout(title="MSFT stock prices", xaxis_title="Date", yaxis_title="Stock price")

fig.show()

### Data splitting

In [ ]:
from sklearn.preprocessing import MinMaxScaler

lag = 15


df = df_GOOGL

values = df['Close'].values

train_len = math.ceil(len(values)*0.8)

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(values.reshape(-1, 1))

train_data = scaled_data[0:train_len, :]
test_data = scaled_data[train_len-lag:, :]

x_train, y_train = [], []

for i in range(60, len(train_data)):
    x_train.append(train_data[i-lag: i, 0])
    y_train.append(train_data[i, 0])

x_train, y_train = np.array(x_train), np.array(y_train)

x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))


x_test, y_test = [], []

for i in range(lag, len(test_data)):
    x_test.append(test_data[i-lag: i, 0])

x_test = np.array(x_test)
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

y_test = values[train_len:]
print(f"X train shape: {x_train.shape}, y train shape: {y_train.shape}, X test shape: {x_test.shape}, y test shape: {y_test.shape}")

# Convert to pytorch tensors

y_train = torch.tensor(y_train, dtype=torch.float32)
y_train = torch.reshape(y_train, [len(x_train), 1])
x_train = torch.tensor(x_train, dtype=torch.float32)

y_test = torch.tensor(y_test, dtype=torch.float32)
y_test = torch.reshape(y_test, [len(x_test), 1])
x_test = torch.tensor(x_test, dtype=torch.float32)
print(f"X train shape: {x_train.shape}, y train shape: {y_train.shape}, X test shape: {x_test.shape}, y test shape: {y_test.shape}")


X train shape: (442, 15, 1), y train shape: (442,), X test shape: (125, 15, 1), y test shape: (125,)
X train shape: torch.Size([442, 15, 1]), y train shape: torch.Size([442, 1]), X test shape: torch.Size([125, 15, 1]), y test shape: torch.Size([125, 1])


### LSTM model initialization

In [254]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out, hn, cn


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model = LSTMModel(input_dim=1, hidden_dim=100, layer_dim=1, output_dim=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

num_epochs = 100
h0, c0 = None, None

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs, h0, c0 = model(x_train, h0, c0)

    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    h0 = h0.detach()
    c0 = c0.detach()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


model.eval()
predicted, _, _ = model(x_train, h0, c0)

original = df[lag:]
time_steps = np.arange(lag, len(df))

predicted[::30] += 0.2 
predicted[::70] -= 0.2



Epoch [10/100], Loss: 0.0279
Epoch [20/100], Loss: 0.0187
Epoch [30/100], Loss: 0.0131
Epoch [40/100], Loss: 0.0051
Epoch [50/100], Loss: 0.0032
Epoch [60/100], Loss: 0.0015
Epoch [70/100], Loss: 0.0013
Epoch [80/100], Loss: 0.0011
Epoch [90/100], Loss: 0.0011
Epoch [100/100], Loss: 0.0010


In [262]:
df = df_GOOGL

data = df['Close']
t = df['Date']


def create_sequences(data, seq_length):
    xs = []
    ys = []
    for i in range(len(data)-seq_length):
        x = data[i:(i+seq_length)]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_length = 10
X, y = create_sequences(data, seq_length)

trainX = torch.tensor(X[:, :, None], dtype=torch.float32)
trainY = torch.tensor(y[:, None], dtype=torch.float32)

In [263]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out, hn, cn

In [ ]:
model = LSTMModel(input_dim=1, hidden_dim=50, layer_dim=1, output_dim=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

num_epochs = 100
h0, c0 = None, None

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs, h0, c0 = model(trainX, h0, c0)

    loss = criterion(outputs, trainY)
    loss.backward()
    optimizer.step()

    h0 = h0.detach()
    c0 = c0.detach()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 14637.0615
Epoch [20/100], Loss: 8786.7119
Epoch [30/100], Loss: 4797.5376
Epoch [40/100], Loss: 2469.9490
